PIRATE 1.5 analysis template: use new schema version 2 campaigns with recorded producer Dcores.
Pre-upgrade results and executed figures remain in the original checkout and backup.


# Gaussian-corruption candidate-pipeline timing

This notebook validates a completed Gaussian-mixture campaign before plotting it. Set `PIRATE_GAUSSIAN_CORRUPTION_RESULTS` to analyze a non-default result directory. Every figure overlays raw trials with a median and interquartile-range (IQR) summary.

Within one trial, each beam reuses the same white noise and Gaussian template at every requested corruption level; only its calibrated amplitude changes. Across trials and beams, independently drawn Gaussian morphologies mean that the same corruption percentage can produce different peakfinder-candidate counts.

In [ ]:
import csv
import hashlib
import json
import os
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import numpy as np
import yaml

SCHEMA_NAME = 'pirate-gaussian-corruption-timing'
SCHEMA_VERSION = 2
EXPECTED_SHAPES = [
    (4096, 128), (1024, 64), (2048, 64), (1024, 64), (1024, 32),
    (2048, 32), (512, 32), (1024, 32), (1024, 16), (2048, 16),
]
EXPECTED_PIXELS_PER_BEAM = 983_040
SUMMARY_METRICS = (
    'total_pixels_above_threshold',
    'total_peakfinder_candidates',
    'peakfinder_survival_fraction',
    'decoder_wall_ms',
    'total_decoded_candidates',
    'grouper_wall_ms',
    'total_grouped_events',
    'decoder_plus_grouper_wall_ms',
    'post_peakfinder_load_fraction',
)
SUMMARY_STATISTICS = ('median', 'minimum', 'maximum', 'q25', 'q75', 'iqr')

configured = os.environ.get('PIRATE_GAUSSIAN_CORRUPTION_RESULTS')
if configured:
    RESULTS_DIR = Path(configured).expanduser()
else:
    candidates = [
        Path('peakfinder_tests/results_gaussian_corruption_timing_pirate15'),
        Path('results_gaussian_corruption_timing_pirate15'),
    ]
    RESULTS_DIR = next((path for path in candidates if path.is_dir()), candidates[0])
PATHS = {name: RESULTS_DIR / name for name in ('trials.csv', 'summary.csv', 'metadata.yaml')}
missing_files = [str(path) for path in PATHS.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'missing Gaussian benchmark results: {missing_files}')

def read_csv_strict(path):
    with path.open(newline='') as stream:
        reader = csv.DictReader(stream)
        fields = tuple(reader.fieldnames or ())
        if not fields or len(fields) != len(set(fields)):
            raise ValueError(f'{path}: missing or duplicate CSV columns')
        rows = list(reader)
    if not rows:
        raise ValueError(f'{path}: table is empty')
    return fields, rows

trial_fields, trial_rows = read_csv_strict(PATHS['trials.csv'])
summary_fields, summary_rows = read_csv_strict(PATHS['summary.csv'])
with PATHS['metadata.yaml'].open() as stream:
    metadata = yaml.safe_load(stream) or {}


In [ ]:
TRIAL_REQUIRED = {
    'schema_version', 'campaign_id', 'trial', 'seed', 'total_beams',
    'beam_batch_size', 'threshold', 'method', 'dm_reach', 'waist_bins',
    'target_corruption_percent', 'achieved_corruption_percent',
    'target_pixels_above_threshold', 'achieved_pixels_above_threshold',
    'pixel_count_error', 'max_abs_beam_pixel_count_error',
    'per_beam_calibration_json', 'total_gaussians', 'gaussians_per_beam_min',
    'gaussians_per_beam_median', 'gaussians_per_beam_max',
    'total_pixels_above_threshold', 'pixels_above_threshold_per_beam_min',
    'pixels_above_threshold_per_beam_median', 'pixels_above_threshold_per_beam_max',
    'total_peakfinder_candidates', 'peakfinder_candidates_per_beam_min',
    'peakfinder_candidates_per_beam_median', 'peakfinder_candidates_per_beam_max',
    'peakfinder_survival_fraction', 'decoder_wall_ms',
    'decoder_candidates_per_second', 'total_decoded_candidates',
    'grouper_wall_ms', 'grouper_candidates_per_second', 'total_grouped_events',
    'grouped_events_per_beam_min', 'grouped_events_per_beam_median',
    'grouped_events_per_beam_max', 'decoder_plus_grouper_wall_ms',
    'post_peakfinder_load_fraction', 'candidate_safety_limit', 'status',
    'failure_reason',
}
SUMMARY_REQUIRED = {
    'schema_version', 'campaign_id', 'target_corruption_percent',
    'recorded_configurations', 'completed_trials', 'skipped_trials', 'failed_trials',
} | {
    f'{metric}_{statistic}'
    for metric in SUMMARY_METRICS
    for statistic in SUMMARY_STATISTICS
}
for name, fields, required in (
    ('trials.csv', trial_fields, TRIAL_REQUIRED),
    ('summary.csv', summary_fields, SUMMARY_REQUIRED),
):
    missing_columns = required - set(fields)
    if missing_columns:
        raise ValueError(f'{name}: missing columns {sorted(missing_columns)}')

def as_int(value, context):
    try:
        result = int(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f'{context}: expected integer, got {value!r}') from exc
    return result

def as_float(value, context, *, allow_missing=False):
    if allow_missing and (value is None or str(value).strip().lower() in {'', 'nan', 'none'}):
        return None
    try:
        result = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f'{context}: expected number, got {value!r}') from exc
    if not np.isfinite(result):
        if allow_missing:
            return None
        raise ValueError(f'{context}: expected finite number, got {value!r}')
    return result

def close(observed, expected, context, *, rtol=1e-9, atol=1e-9):
    if not np.isclose(float(observed), float(expected), rtol=rtol, atol=atol):
        raise ValueError(f'{context}: observed {observed!r}, expected {expected!r}')

REQUIRED_METADATA = {
    'schema_name', 'schema_version', 'campaign_id', 'campaign_signature',
    'campaign', 'plan', 'corruption_model', 'peakfinder', 'streaming_state',
    'decoder_timing', 'grouper_timing', 'safety_limit', 'environment',
    'csv_schemas', 'campaign_completeness',
}
if not REQUIRED_METADATA <= set(metadata):
    raise ValueError(f'metadata.yaml: missing sections {sorted(REQUIRED_METADATA - set(metadata))}')
if metadata['schema_name'] != SCHEMA_NAME or as_int(metadata['schema_version'], 'metadata schema') != SCHEMA_VERSION:
    raise ValueError('metadata.yaml: unsupported schema')
campaign_id = metadata['campaign_id']
if not isinstance(campaign_id, str) or not campaign_id:
    raise ValueError('metadata.yaml: campaign_id must be a non-empty string')
signature = metadata['campaign_signature']
if not isinstance(signature, str) or len(signature) != 64:
    raise ValueError('metadata.yaml: campaign_signature must be a SHA-256 hex digest')
signature_payload = metadata.get('campaign_signature_payload')
if not isinstance(signature_payload, dict):
    raise ValueError('metadata.yaml: campaign_signature_payload must be a mapping')
computed_signature = hashlib.sha256(
    json.dumps(signature_payload, sort_keys=True, separators=(',', ':'), ensure_ascii=True).encode('utf-8')
).hexdigest()
if computed_signature != signature:
    raise ValueError('metadata.yaml: campaign signature does not match its payload')

plan = metadata['plan']
if plan.get('argmax_encoding') != 'pirate-1.5:t8-p8-m8-mu8':
    raise ValueError('metadata.yaml: missing or incompatible token encoding')
dcores = plan.get('dcores')
tree_plan = plan.get('tree_plan')
if (not isinstance(dcores, list) or len(dcores) != len(EXPECTED_SHAPES)
        or any(isinstance(v, bool) or not isinstance(v, int)
               or v < 1 or v > 256 or v & (v - 1) for v in dcores)):
    raise ValueError('metadata.yaml: invalid producer Dcores')
if not isinstance(tree_plan, list) or len(tree_plan) != len(dcores):
    raise ValueError('metadata.yaml: missing per-tree token geometry')
if signature_payload.get('dcores') != dcores or signature_payload.get('argmax_encoding') != plan['argmax_encoding']:
    raise ValueError('metadata.yaml: producer metadata disagrees with campaign signature')
signed_trees = signature_payload.get('token_geometry')
if not isinstance(signed_trees, list) or len(signed_trees) != len(dcores):
    raise ValueError('metadata.yaml: missing signed token geometry')
for dcore, tree, signed_tree in zip(dcores, tree_plan, signed_trees):
    if (tree.get('dcore') != dcore or int(tree.get('token_dout', 0)) < 1
            or int(tree['token_dout']) % dcore
            or not isinstance(signed_tree, dict)
            or any(tree.get(key) != value for key, value in signed_tree.items())):
        raise ValueError('metadata.yaml: inconsistent producer token geometry')
shapes = [tuple(map(int, shape)) for shape in plan['tree_shapes_ndm_ntime']]
if shapes != EXPECTED_SHAPES:
    raise ValueError(f'metadata.yaml: unexpected plan-derived shapes {shapes}')
pixels_per_beam = as_int(plan['pixels_per_beam'], 'pixels_per_beam')
if pixels_per_beam != EXPECTED_PIXELS_PER_BEAM:
    raise ValueError('metadata.yaml: pixels_per_beam is not 983,040')
chunk_duration_ms = as_float(plan['chunk_duration_ms'], 'metadata chunk_duration_ms')
config_document = plan['config_document']
expected_chunk_duration_ms = (
    as_float(config_document['time_sample_ms'], 'config time_sample_ms')
    * as_int(config_document['time_samples_per_chunk'], 'config time samples per chunk')
)
close(chunk_duration_ms, expected_chunk_duration_ms, 'metadata chunk duration')

campaign = metadata['campaign']
total_beams = as_int(campaign['total_beams'], 'metadata total_beams')
beam_batch_size = as_int(campaign['beam_batch_size'], 'metadata beam_batch_size')
trial_count = as_int(campaign['trials'], 'metadata trials')
target_grid = tuple(float(value) for value in campaign['percentage_grid'])
if total_beams < 1 or beam_batch_size < 1 or trial_count < 1 or not target_grid:
    raise ValueError('metadata.yaml: invalid campaign dimensions')
if len(target_grid) != len(set(target_grid)) or any(value < 0 for value in target_grid):
    raise ValueError('metadata.yaml: invalid corruption-percentage grid')
total_pixels_per_trial = as_int(campaign['total_pixels_per_trial'], 'metadata total_pixels_per_trial')
if total_pixels_per_trial != total_beams * pixels_per_beam:
    raise ValueError('metadata.yaml: total_pixels_per_trial is inconsistent')
if metadata['corruption_model'].get('name') != 'gaussian_mixture':
    raise ValueError('metadata.yaml: corruption model is not gaussian_mixture')
calibration_tolerance = as_int(
    metadata['corruption_model']['calibration']['pixel_tolerance_per_beam'],
    'metadata calibration tolerance',
)
if calibration_tolerance < 0:
    raise ValueError('metadata.yaml: calibration tolerance must be non-negative')
candidate_safety_limit = as_int(
    metadata['safety_limit']['candidate_limit'], 'metadata candidate safety limit'
)
unsafe_override = metadata['safety_limit']['unsafe_override']
if candidate_safety_limit < 1 or not isinstance(unsafe_override, bool):
    raise ValueError('metadata.yaml: invalid candidate safety policy')

csv_schemas = metadata['csv_schemas']
for filename, observed_fields in (('trials.csv', trial_fields), ('summary.csv', summary_fields)):
    if csv_schemas.get(filename) != list(observed_fields):
        raise ValueError(f'metadata.yaml: schema for {filename} disagrees with the file')
completeness = metadata['campaign_completeness']
complete_flag = completeness['complete']
if complete_flag is not True:
    raise ValueError('metadata.yaml: campaign is incomplete; finish or resume it before analysis')
expected_configurations = trial_count * len(target_grid)
if as_int(completeness['expected_configurations'], 'expected configurations') != expected_configurations:
    raise ValueError('metadata.yaml: expected configuration count disagrees with campaign grid')
if as_int(completeness['recorded_configurations'], 'recorded configurations') != expected_configurations:
    raise ValueError('metadata.yaml: not all configurations were recorded')

rows_by_key = {}
valid_rows = []
grouped_rows = []
grouped_statuses = {'completed'}
for lineno, row in enumerate(trial_rows, start=2):
    context = f'trials.csv:{lineno}'
    if as_int(row['schema_version'], context + ':schema_version') != SCHEMA_VERSION or row['campaign_id'] != campaign_id:
        raise ValueError(f'{context}: schema or campaign mismatch')
    trial = as_int(row['trial'], context + ':trial')
    target = as_float(row['target_corruption_percent'], context + ':target')
    matching_targets = [value for value in target_grid if np.isclose(target, value, rtol=0, atol=1e-12)]
    if not matching_targets or not 0 <= trial < trial_count:
        raise ValueError(f'{context}: row lies outside the campaign grid')
    target = matching_targets[0]
    key = (trial, target)
    if key in rows_by_key:
        raise ValueError(f'{context}: duplicate trial/target key {key}')
    rows_by_key[key] = row
    if as_int(row['total_beams'], context + ':total_beams') != total_beams:
        raise ValueError(f'{context}: total_beams disagrees with metadata')
    if as_int(row['beam_batch_size'], context + ':beam_batch_size') != beam_batch_size:
        raise ValueError(f'{context}: beam_batch_size disagrees with metadata')
    status = row['status'].strip()
    if status.startswith('failed_'):
        continue
    row_safety_limit = as_int(row['candidate_safety_limit'], context + ':candidate safety limit')
    if row_safety_limit != candidate_safety_limit:
        raise ValueError(f'{context}: candidate safety limit disagrees with metadata')
    try:
        beam_calibrations = json.loads(row['per_beam_calibration_json'])
    except (TypeError, json.JSONDecodeError) as exc:
        raise ValueError(f'{context}: malformed per_beam_calibration_json') from exc
    if not isinstance(beam_calibrations, list) or len(beam_calibrations) != total_beams:
        raise ValueError(f'{context}: calibration JSON must have one row per beam')
    expected_target_per_beam = int(np.rint(target / 100.0 * pixels_per_beam))
    achieved_by_beam = []
    errors_by_beam = []
    observed_beam_ids = []
    for beam_index, item in enumerate(beam_calibrations):
        if not isinstance(item, dict):
            raise ValueError(f'{context}: calibration entry {beam_index} is not a mapping')
        beam_id = as_int(item.get('beam_id'), context + f':calibration[{beam_index}].beam_id')
        target_count = as_int(item.get('target_count'), context + f':calibration[{beam_index}].target_count')
        achieved_count = as_int(item.get('achieved_count'), context + f':calibration[{beam_index}].achieved_count')
        count_error = as_int(item.get('pixel_count_error'), context + f':calibration[{beam_index}].pixel_count_error')
        scale = as_float(item.get('scale'), context + f':calibration[{beam_index}].scale')
        evaluations = as_int(item.get('calibration_evaluations'), context + f':calibration[{beam_index}].evaluations')
        if target_count != expected_target_per_beam or achieved_count < 0:
            raise ValueError(f'{context}: calibration entry {beam_index} has inconsistent counts')
        if count_error != achieved_count - target_count or abs(count_error) > calibration_tolerance:
            raise ValueError(f'{context}: calibration entry {beam_index} violates pixel tolerance')
        if scale < 0 or evaluations < 0 or (target_count > 0 and evaluations < 1):
            raise ValueError(f'{context}: calibration entry {beam_index} has invalid scale/evaluations')
        observed_beam_ids.append(beam_id)
        achieved_by_beam.append(achieved_count)
        errors_by_beam.append(count_error)
    if sorted(observed_beam_ids) != list(range(total_beams)):
        raise ValueError(f'{context}: calibration JSON has invalid or duplicate beam IDs')
    above = as_int(row['total_pixels_above_threshold'], context + ':pixels above threshold')
    candidates = as_int(row['total_peakfinder_candidates'], context + ':peakfinder candidates')
    decoded = as_int(row['total_decoded_candidates'], context + ':decoded candidates')
    if min(above, candidates, decoded) < 0 or decoded != candidates:
        raise ValueError(f'{context}: negative count or decoder lost candidates')
    target_total = expected_target_per_beam * total_beams
    if as_int(row['target_pixels_above_threshold'], context + ':target pixels') != target_total:
        raise ValueError(f'{context}: target pixel total is inconsistent')
    if as_int(row['achieved_pixels_above_threshold'], context + ':achieved pixels') != above or sum(achieved_by_beam) != above:
        raise ValueError(f'{context}: achieved pixel totals are inconsistent')
    aggregate_error = above - target_total
    if as_int(row['pixel_count_error'], context + ':pixel count error') != aggregate_error or sum(errors_by_beam) != aggregate_error:
        raise ValueError(f'{context}: aggregate pixel error is inconsistent')
    maximum_error = max((abs(value) for value in errors_by_beam), default=0)
    if as_int(row['max_abs_beam_pixel_count_error'], context + ':maximum beam error') != maximum_error:
        raise ValueError(f'{context}: maximum per-beam pixel error is inconsistent')
    for suffix, expected_value in (
        ('min', np.min(achieved_by_beam)), ('median', np.median(achieved_by_beam)),
        ('max', np.max(achieved_by_beam)),
    ):
        close(row[f'pixels_above_threshold_per_beam_{suffix}'], expected_value, context + f':pixels per beam {suffix}')
    achieved = as_float(row['achieved_corruption_percent'], context + ':achieved percent')
    close(achieved, 100.0 * above / (total_beams * pixels_per_beam), context + ':achieved percent', atol=5e-12)
    survival = as_float(row['peakfinder_survival_fraction'], context + ':survival')
    close(survival, candidates / above if above else 0.0, context + ':survival')
    decoder_ms = as_float(row['decoder_wall_ms'], context + ':decoder wall')
    if decoder_ms < 0:
        raise ValueError(f'{context}: negative decoder time')
    if np.isclose(target, 0.0, rtol=0, atol=1e-12) and any((above, candidates, decoded)):
        raise ValueError(f'{context}: zero corruption produced above-threshold pixels or candidates')
    valid_rows.append(row)
    grouper_ms = as_float(row['grouper_wall_ms'], context + ':grouper wall', allow_missing=True)
    events = as_float(row['total_grouped_events'], context + ':events', allow_missing=True)
    combined = as_float(row['decoder_plus_grouper_wall_ms'], context + ':combined wall', allow_missing=True)
    load = as_float(row['post_peakfinder_load_fraction'], context + ':load', allow_missing=True)
    if status in grouped_statuses:
        if None in (grouper_ms, events, combined, load) or min(grouper_ms, events, combined, load) < 0:
            raise ValueError(f'{context}: completed grouping row has missing or negative results')
        if decoded > candidate_safety_limit and not unsafe_override:
            raise ValueError(f'{context}: completed grouping exceeds the safety limit without override')
        close(combined, decoder_ms + grouper_ms, context + ':combined wall')
        close(load, combined / chunk_duration_ms, context + ':post-peakfinder load')
        grouped_rows.append(row)
    elif status == 'skipped_candidate_safety_limit':
        if unsafe_override or decoded <= candidate_safety_limit:
            raise ValueError(f'{context}: safety skip is inconsistent with the configured limit/override')
        for field in (
            'grouper_wall_ms', 'grouper_candidates_per_second', 'total_grouped_events',
            'grouped_events_per_beam_min', 'grouped_events_per_beam_median',
            'grouped_events_per_beam_max', 'decoder_plus_grouper_wall_ms',
            'post_peakfinder_load_fraction',
        ):
            if row[field].strip():
                raise ValueError(f'{context}: safety-skipped {field} must be empty')
    else:
        raise ValueError(f'{context}: unsupported status {status!r}')

expected_keys = {(trial, target) for trial in range(trial_count) for target in target_grid}
if set(rows_by_key) != expected_keys:
    raise ValueError('trials.csv does not contain every requested trial/percentage exactly once')
if not valid_rows:
    raise ValueError('trials.csv contains no successfully calibrated configurations')

summary_by_target = {}
for lineno, row in enumerate(summary_rows, start=2):
    context = f'summary.csv:{lineno}'
    if as_int(row['schema_version'], context + ':schema_version') != SCHEMA_VERSION or row['campaign_id'] != campaign_id:
        raise ValueError(f'{context}: schema or campaign mismatch')
    target = as_float(row['target_corruption_percent'], context + ':target')
    matching = [value for value in target_grid if np.isclose(target, value, rtol=0, atol=1e-12)]
    if not matching or matching[0] in summary_by_target:
        raise ValueError(f'{context}: unexpected or duplicate target percentage')
    summary_by_target[matching[0]] = row
if set(summary_by_target) != set(target_grid):
    raise ValueError('summary.csv does not contain every requested corruption percentage')

def finite_values(rows, field):
    values = [as_float(row[field], field, allow_missing=True) for row in rows]
    return np.asarray([value for value in values if value is not None], dtype=float)

for target, summary in summary_by_target.items():
    matching_rows = [
        row for row in trial_rows
        if np.isclose(float(row['target_corruption_percent']), target, rtol=0, atol=1e-12)
    ]
    expected_counts = {
        'recorded_configurations': len(matching_rows),
        'skipped_trials': sum(row['status'].strip() == 'skipped_candidate_safety_limit' for row in matching_rows),
        'failed_trials': sum(row['status'].strip().startswith('failed_') for row in matching_rows),
    }
    for field, expected_count in expected_counts.items():
        if as_int(summary[field], f'summary {target}:{field}') != expected_count:
            raise ValueError(f'summary {target}: {field} disagrees with trials.csv')
    source_rows = [
        row for row in valid_rows
        if row['status'].strip() in grouped_statuses
        and np.isclose(float(row['target_corruption_percent']), target, rtol=0, atol=1e-12)
    ]
    if as_int(summary['completed_trials'], f'summary {target}:completed_trials') != len(source_rows):
        raise ValueError(f'summary {target}: completed_trials disagrees with trials.csv')
    for metric in SUMMARY_METRICS:
        values = finite_values(source_rows, metric)
        recorded = {stat: as_float(summary[f'{metric}_{stat}'], f'summary {target}:{metric}_{stat}', allow_missing=True) for stat in SUMMARY_STATISTICS}
        if values.size == 0:
            if any(value is not None for value in recorded.values()):
                raise ValueError(f'summary {target}:{metric} should be empty')
            continue
        expected = {
            'median': np.median(values), 'minimum': np.min(values), 'maximum': np.max(values),
            'q25': np.quantile(values, 0.25), 'q75': np.quantile(values, 0.75),
        }
        expected['iqr'] = expected['q75'] - expected['q25']
        for statistic, value in expected.items():
            if recorded[statistic] is None:
                raise ValueError(f'summary {target}:{metric}_{statistic} is missing')
            close(recorded[statistic], value, f'summary {target}:{metric}_{statistic}', rtol=1e-8, atol=1e-10)

print(f'Validated {len(trial_rows)} configurations in {RESULTS_DIR}')


In [ ]:
targets = np.asarray(sorted(target_grid), dtype=float)

def paired_arrays(rows, xfield, yfield):
    pairs = []
    for row in rows:
        x = as_float(row[xfield], xfield, allow_missing=True)
        y = as_float(row[yfield], yfield, allow_missing=True)
        if x is not None and y is not None:
            pairs.append((x, y))
    if not pairs:
        raise ValueError(f'no finite pairs for {xfield} versus {yfield}')
    return np.asarray(pairs, dtype=float).T

def target_quantiles(rows, field):
    result = []
    for target in targets:
        selected = [row for row in rows if np.isclose(float(row['target_corruption_percent']), target, rtol=0, atol=1e-12)]
        values = finite_values(selected, field)
        if values.size:
            result.append((target, np.quantile(values, 0.25), np.median(values), np.quantile(values, 0.75)))
    return np.asarray(result, dtype=float).T

def relationship_quantiles(rows, xfield, yfield):
    result = []
    for target in targets:
        selected = [row for row in rows if np.isclose(float(row['target_corruption_percent']), target, rtol=0, atol=1e-12)]
        pairs = []
        for row in selected:
            xvalue = as_float(row[xfield], xfield, allow_missing=True)
            yvalue = as_float(row[yfield], yfield, allow_missing=True)
            if xvalue is not None and yvalue is not None:
                pairs.append((xvalue, yvalue))
        if not pairs:
            continue
        x, y = np.asarray(pairs, dtype=float).T
        result.append((
            np.quantile(x, 0.25), np.median(x), np.quantile(x, 0.75),
            np.quantile(y, 0.25), np.median(y), np.quantile(y, 0.75),
        ))
    return np.asarray(result, dtype=float).T

def draw_target_summary(ax, rows, field, *, color='tab:orange', label='median and IQR'):
    x, q25, median, q75 = target_quantiles(rows, field)
    ax.plot(x, median, color=color, marker='o', linewidth=2, label=label)
    ax.fill_between(x, q25, q75, color=color, alpha=0.22)

def draw_relationship_summary(ax, rows, xfield, yfield):
    xq25, xmed, xq75, yq25, ymed, yq75 = relationship_quantiles(rows, xfield, yfield)
    ax.errorbar(
        xmed, ymed, xerr=np.vstack((xmed - xq25, xq75 - xmed)),
        yerr=np.vstack((ymed - yq25, yq75 - ymed)), fmt='o-',
        color='black', linewidth=1.6, capsize=3, label='median with IQR', zorder=4,
    )

def use_zero_safe_log(axis, values):
    positives = np.asarray(values, dtype=float)
    positives = positives[positives > 0]
    if positives.size:
        axis.set_xscale('symlog', linthresh=max(float(np.min(positives)) / 2.0, 1e-12))

plt.rcParams.update({'figure.figsize': (9.5, 6.0), 'axes.grid': True, 'grid.alpha': 0.25})


The paired corruption levels control pixel occupancy, not morphology. Candidate-count scatter at a fixed percentage is therefore expected: each trial and beam has a different Gaussian count, location, width, orientation/correlation, and relative-amplitude draw.

In [ ]:
# 1. Achieved versus target corruption percentage
fig, ax = plt.subplots()
raw_target, raw_achieved = paired_arrays(valid_rows, 'target_corruption_percent', 'achieved_corruption_percent')
ax.scatter(raw_target, raw_achieved, s=22, alpha=0.35, label='raw trials')
draw_target_summary(ax, valid_rows, 'achieved_corruption_percent')
limit = max(float(np.max(raw_target)), float(np.max(raw_achieved)), 1e-6)
ax.plot([0, limit], [0, limit], linestyle='--', color='0.35', label='target = achieved')
use_zero_safe_log(ax, raw_target)
positive_y = raw_achieved[raw_achieved > 0]
if positive_y.size:
    ax.set_yscale('symlog', linthresh=max(float(np.min(positive_y)) / 2.0, 1e-12))
ax.set_xlabel('target corruption (%)')
ax.set_ylabel('achieved corruption after float16 cast (%)')
ax.set_title('1. Per-beam calibrated corruption')
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# 2. Total peakfinder candidates versus pixels above threshold
fig, ax = plt.subplots()
x, y = paired_arrays(valid_rows, 'total_pixels_above_threshold', 'total_peakfinder_candidates')
ax.scatter(x, y, s=22, alpha=0.35, label='raw trials')
draw_relationship_summary(ax, valid_rows, 'total_pixels_above_threshold', 'total_peakfinder_candidates')
use_zero_safe_log(ax, x)
positive_y = y[y > 0]
if positive_y.size:
    ax.set_yscale('symlog', linthresh=max(float(np.min(positive_y)) / 2.0, 1.0))
ax.set_xlabel('total pixels at or above threshold')
ax.set_ylabel('total surviving peakfinder candidates')
ax.set_title('2. Bowtie peakfinder compaction')
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# 3. Peakfinder survival fraction versus corruption percentage
fig, ax = plt.subplots()
x, y = paired_arrays(valid_rows, 'target_corruption_percent', 'peakfinder_survival_fraction')
ax.scatter(x, y, s=22, alpha=0.35, label='raw trials')
draw_target_summary(ax, valid_rows, 'peakfinder_survival_fraction')
use_zero_safe_log(ax, x)
positive_y = y[y > 0]
if positive_y.size:
    ax.set_yscale('symlog', linthresh=max(float(np.min(positive_y)) / 2.0, 1e-12))
ax.set_xlabel('target corruption (%)')
ax.set_ylabel('candidates / pixels above threshold')
ax.set_title('3. Peakfinder survival fraction (zero-input convention: 0)')
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# 4. Peakfinder candidates versus Gaussian count, coloured by corruption percentage
fig, ax = plt.subplots()
x, y = paired_arrays(valid_rows, 'total_gaussians', 'total_peakfinder_candidates')
colors = np.asarray([float(row['target_corruption_percent']) for row in valid_rows], dtype=float)
normalizer = Normalize(vmin=float(np.min(colors)), vmax=float(np.max(colors)) or 1.0)
points = ax.scatter(x, y, c=colors, norm=normalizer, cmap='viridis', s=28, alpha=0.55, label='raw trials')
draw_relationship_summary(ax, valid_rows, 'total_gaussians', 'total_peakfinder_candidates')
if np.all(x > 0):
    ax.set_xscale('log')
positive_y = y[y > 0]
if positive_y.size:
    ax.set_yscale('symlog', linthresh=max(float(np.min(positive_y)) / 2.0, 1.0))
fig.colorbar(points, ax=ax, label='target corruption (%)')
ax.set_xlabel('total Gaussian components')
ax.set_ylabel('total surviving peakfinder candidates')
ax.set_title('4. Morphology variation at controlled occupancy')
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# 5. Decoder wall time versus decoded candidate count
fig, ax = plt.subplots()
x, y = paired_arrays(valid_rows, 'total_decoded_candidates', 'decoder_wall_ms')
ax.scatter(x, y, s=22, alpha=0.35, label='raw trials')
draw_relationship_summary(ax, valid_rows, 'total_decoded_candidates', 'decoder_wall_ms')
use_zero_safe_log(ax, x)
ax.set_xlabel('decoded candidate count')
ax.set_ylabel('decoder synchronized wall time (ms)')
ax.set_title('5. Production GPU decoder scaling')
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# 6. Grouper wall time versus decoded candidate count
fig, ax = plt.subplots()
x, y = paired_arrays(grouped_rows, 'total_decoded_candidates', 'grouper_wall_ms')
ax.scatter(x, y, s=22, alpha=0.35, label='raw grouped trials')
draw_relationship_summary(ax, grouped_rows, 'total_decoded_candidates', 'grouper_wall_ms')
use_zero_safe_log(ax, x)
ax.set_xlabel('decoded candidate count')
ax.set_ylabel('grouper synchronized wall time (ms)')
ax.set_title('6. Current production multi-beam grouper scaling')
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# 7. Total grouped events versus decoded candidate count
fig, ax = plt.subplots()
x, y = paired_arrays(grouped_rows, 'total_decoded_candidates', 'total_grouped_events')
ax.scatter(x, y, s=22, alpha=0.35, label='raw grouped trials')
draw_relationship_summary(ax, grouped_rows, 'total_decoded_candidates', 'total_grouped_events')
use_zero_safe_log(ax, x)
positive_y = y[y > 0]
if positive_y.size:
    ax.set_yscale('symlog', linthresh=max(float(np.min(positive_y)) / 2.0, 1.0))
ax.set_xlabel('decoded candidate count')
ax.set_ylabel('final grouped event count')
ax.set_title('7. Candidate-to-event reduction')
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# 8. Grouper wall time versus final grouped event count
fig, ax = plt.subplots()
x, y = paired_arrays(grouped_rows, 'total_grouped_events', 'grouper_wall_ms')
ax.scatter(x, y, s=22, alpha=0.35, label='raw grouped trials')
draw_relationship_summary(ax, grouped_rows, 'total_grouped_events', 'grouper_wall_ms')
use_zero_safe_log(ax, x)
ax.set_xlabel('final grouped event count')
ax.set_ylabel('grouper synchronized wall time (ms)')
ax.set_title('8. Grouper cost versus produced events')
ax.legend()
fig.tight_layout()
plt.show()


In [ ]:
# 9. Decoder-plus-grouper time and post-peakfinder real-time load
fig, time_ax = plt.subplots()
load_ax = time_ax.twinx()
raw_x, raw_time = paired_arrays(grouped_rows, 'target_corruption_percent', 'decoder_plus_grouper_wall_ms')
_, raw_load = paired_arrays(grouped_rows, 'target_corruption_percent', 'post_peakfinder_load_fraction')
time_ax.scatter(raw_x, raw_time, s=22, alpha=0.3, color='tab:blue', label='raw combined time')
load_ax.scatter(raw_x, 100.0 * raw_load, s=22, alpha=0.3, color='tab:red', marker='x', label='raw real-time load')
draw_target_summary(time_ax, grouped_rows, 'decoder_plus_grouper_wall_ms', color='tab:blue', label='time median and IQR')
load_x, load_q25, load_median, load_q75 = target_quantiles(grouped_rows, 'post_peakfinder_load_fraction')
load_ax.plot(load_x, 100.0 * load_median, color='tab:red', marker='s', linewidth=2, label='load median and IQR')
load_ax.fill_between(load_x, 100.0 * load_q25, 100.0 * load_q75, color='tab:red', alpha=0.16)
use_zero_safe_log(time_ax, raw_x)
time_ax.set_xlabel('target corruption (%)')
time_ax.set_ylabel('decoder + grouper synchronized stage sum (ms)', color='tab:blue')
load_ax.set_ylabel(
    f'post-peakfinder real-time load (% of {chunk_duration_ms:g} ms)',
    color='tab:red',
)
time_ax.set_title('9. Post-peakfinder time and real-time load')
handles_a, labels_a = time_ax.get_legend_handles_labels()
handles_b, labels_b = load_ax.get_legend_handles_labels()
time_ax.legend(handles_a + handles_b, labels_a + labels_b, loc='best')
fig.tight_layout()
plt.show()
